# TN1 phần B — TCN và DS-TCN

Chạy **song song** với `TN1_LSTM.ipynb` ở một phiên Colab khác. Hai notebook độc lập hoàn toàn, không cần chờ nhau.

| notebook | chạy gì | thời gian |
|---|---|---|
| TN1_LSTM.ipynb | LSTM: 4 fold CV, rồi 3 seed test GHIJ | ~3 giờ |
| **TN1_TCN_DSTCN.ipynb** ← đang mở | TCN-64 và DS-TCN-64: 4 fold CV mỗi cái | ~2.2 giờ |
| TN1.ipynb | gộp kết quả, so sánh, chạy GHIJ cho kiến trúc thắng | ~1.5 giờ |

## Hai kiến trúc

| model | kênh | tham số | so LSTM |
|---|---|---|---|
| TCN | 64 | 151.513 | −90% |
| DS-TCN | 64 | 56.281 | −96% |

Khác nhau **đúng một chỗ** — phép tích chập trong mỗi khối:

```python
# TCN thường
nn.Conv1d(64, 64, kernel_size=3, dilation=d)

# DS-TCN, tách làm hai bước (Howard et al. 2017, mục 3.1)
nn.Conv1d(64, 64, 3, dilation=d, groups=64)   # depthwise: mỗi kênh một bộ lọc
nn.Conv1d(64, 64, 1)                          # pointwise: chỉ trộn kênh
```

Mọi thứ khác giống hệt: `kernel=3`, `n_blocks=6`, hai tầng conv mỗi khối, `dropout=0.0`, BatchNorm, ReLU, nối tắt. Trích dẫn từng tham số ở [`docs/THAM_CHIEU.md`](../docs/THAM_CHIEU.md).

## Giao thức

Bốn fold cố định trên tám người `A B C D E F K L`, dùng y nguyên cho mọi thí nghiệm:

```
val_AB   train C D E F K L    chấm A B
val_CE   train A B D F K L    chấm C E
val_DF   train A B C E K L    chấm D F
val_KL   train A B C D E F    chấm K L
```

Cấu hình giữ nguyên như MobiVital công bố: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr_threshold` 0.9, không RevIN. Một seed.

Điểm chấm trên **buổi ghi thô**, model tự chọn kênh, không nhìn nhịp thở thật.


## 1. Chuẩn bị Colab


Mount Drive để lấy lại cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Tải mã nguồn rồi vào thư mục đó. `setup_colab.py` clone MobiVital và ghim commit `4319731d` — `src/mobivital_reference.py` mượn sáu hàm từ repo họ.


In [ ]:
# Phải clone repo trước, vì setup_colab.py nằm bên trong chính repo đó.
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py


Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB — chỉ train trên cửa sổ đã cắt, chấm trên `by_user/*.npz`.


In [ ]:
!python scripts/restore_processed_data_on_drive.py


## 2. TCN-64 — 4 fold CV

Khoảng **1 giờ**.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model tcn --channels 64


## 3. DS-TCN-64 — 4 fold CV

Khoảng **1.2 giờ**.


In [ ]:
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 64


## 4. Xem nhanh

So hai kiến trúc với nhau. LSTM chưa có ở phiên này nên chưa so được với mốc — việc đó làm ở `TN1_final_evaluation.ipynb` sau khi gộp.


In [ ]:
!python scripts/compare_cv.py --experiment tn1


## 5. Cất kết quả

Nén ra tên riêng `tn1_tcn.zip` để **không đè** tệp của phiên chạy LSTM.


In [ ]:
!cd runs && zip -qr /content/drive/MyDrive/mobivital/tn1_tcn.zip tn1 summary.csv
!ls -la /content/drive/MyDrive/mobivital/
